# Set up Selenium

*INFO 4617 Web Data Science · Week 8 · before Wednesday's lab · about 15 minutes*

Wednesday's lab drives a real Chrome window from Python, with a library called **Selenium**. On a computer that has only Anaconda, chapter 8's browser code **won't run as it is**:

- **Anaconda doesn't include Selenium**, so `import selenium` fails.
- **Selenium needs a driver**: a small program, matched to your version of Chrome, through which Python controls the browser. Selenium downloads it the first time, about 10 MB. If you have no Chrome, it downloads **Chrome for Testing** too, about 190 MB. A slow network can stall that download, and some networks block it.
- **An old driver can get in the way.** One that other software left on your computer can be picked instead of the matching one, and then Chrome doesn't start.

This notebook goes through those one at a time. It fixes what it can and tells you what to do about the rest. You don't need the course's `webdata` environment: the notebook works in whatever Python your Jupyter runs, Anaconda's included.

**How to use it.** Run each cell in order with **Shift+Enter**, and read the last line it prints:

- **OK**: go on to the next cell.
- **FIX**: do what it says, then run that cell again. Don't go on until it says OK.

Do it on a fast connection, not on the lab's Wi-Fi just before class. If a FIX doesn't work, bring everything that cell printed to Wednesday's lab.

## 1 · Install Selenium

This cell checks whether the Python that runs this notebook has Selenium 4.20 or later. If it doesn't, the cell installs Selenium with `pip`, which downloads it from the Python Package Index (about 12 MB). If Selenium is there already, it installs nothing.

While it installs, `pip` may print warnings, even a line starting with ERROR about other packages, and a note that you may need to restart the kernel. None of that matters if the cell's last line says OK.

In [ ]:
import importlib
import sys
from importlib.metadata import PackageNotFoundError, version


def new_enough(installed):
    # True for Selenium 4.20 or later; steps 3 and 4 need it
    return installed is not None and tuple(int(n) for n in installed.split(".")[:2]) >= (4, 20)


def selenium_version():
    try:
        return version("selenium")
    except PackageNotFoundError:
        return None


print("This notebook runs", sys.executable)
if not new_enough(selenium_version()):
    print("Installing Selenium. This can take a minute.")
    %pip install --progress-bar off "selenium>=4.20"
    importlib.invalidate_caches()  # so that Python sees the package pip just added

if new_enough(selenium_version()):
    print("OK: Selenium", selenium_version())
else:
    print("FIX: Selenium didn't install. Look above for lines that start with ERROR.")
    print('If one says "Access is denied" or "Permission denied", run this in a new cell:')
    print('    %pip install --user "selenium>=4.20"')
    print("then run this cell again. Otherwise, check your connection and run this cell again.")

## 2 · Settings

Selenium comes with **Selenium Manager**, a small program that finds your Chrome and downloads the driver that matches it. Selenium Manager reads the settings below each time it runs, so this cell makes them first. They last until the kernel restarts. Chapter 8's notebook makes the same settings in its step 1.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

os.environ["SE_SKIP_DRIVER_IN_PATH"] = "true"  # ignore old drivers left on your computer
os.environ["SE_AVOID_STATS"] = "true"          # send no usage statistics
# os.environ["SE_PROXY"] = "http://proxy.example.edu:3128"  # only on a network that needs a proxy

# Only in the course's webdata environment, whose Selenium keeps Selenium Manager outside the
# package: `conda activate webdata` tells Selenium where it is, and so does this.
manager = Path(sys.prefix) / ("Scripts/selenium-manager.exe" if os.name == "nt" else "bin/selenium-manager")
if "SE_MANAGER_PATH" not in os.environ and manager.is_file():
    os.environ["SE_MANAGER_PATH"] = str(manager)

if importlib.util.find_spec("selenium") is None:
    print("FIX: Selenium isn't installed yet. Go back to step 1.")
else:
    print("OK: settings made. Selenium Manager:", os.environ.get("SE_MANAGER_PATH", "inside the selenium package"))

## 3 · Find Chrome and get its driver

This cell runs Selenium Manager on its own, before any browser starts, and shows its log. The first time, it downloads, so give it a few minutes.

The log has a pink background. In Jupyter that only means the lines went to Python's error stream; it doesn't mean something failed. Read the log from the top:

- `Detected browser: chrome` and a version: you have Chrome, and only the driver downloads.
- `chrome not found in the system`: you have no Chrome, so Chrome for Testing downloads too.
- `Driver path:` and `Browser path:`: the two files it settled on. What it downloads goes into its cache folder, so later runs download nothing.

In [ ]:
import logging
from logging.handlers import BufferingHandler

from selenium.webdriver.common.selenium_manager import SeleniumManager

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")
log = logging.getLogger("selenium.webdriver.common.selenium_manager")
kept = BufferingHandler(capacity=1000)  # keeps Selenium Manager's warnings for step 4
kept.setLevel(logging.WARNING)

log.addHandler(kept)
log.setLevel(logging.DEBUG)  # show each decision Selenium Manager makes
try:
    paths = SeleniumManager().binary_paths(["--browser", "chrome"])
    print("OK: Selenium Manager found a browser and a driver. Step 4 checks them.")
except Exception as error:
    paths = None
    print("FIX: Selenium Manager stopped:", error)
    if "Selenium Manager binary" in str(error):
        print("Selenium couldn't find Selenium Manager: run step 2, then this cell again.")
    else:
        print('If it says "error sending request", or mentions a connection or a timeout, your network')
        print("blocked the download. Try another network, such as home Wi-Fi, and run this cell again.")
finally:
    log.setLevel(logging.WARNING)  # from here on, show only its warnings
    log.removeHandler(kept)
manager_warnings = [record.getMessage() for record in kept.buffer]

## 4 · Check what it found

Start the browser only when this cell says OK. That means both files are there, and the driver is one that Selenium Manager matched to your browser, not an old one from somewhere else on your computer.

In [ ]:
import os
from pathlib import Path

cache = Path(os.environ.get("SE_CACHE_PATH", Path.home() / ".cache" / "selenium"))  # Selenium Manager's
problems = []
if "paths" not in globals():
    problems.append("step 3 hasn't finished. Run it, do what it says, then run this cell again.")
elif paths is None:
    problems.append("step 3 didn't finish. Do what it says, then run step 3 and this cell again.")
else:
    driver_path = Path(paths["driver_path"])
    browser_path = Path(paths["browser_path"])
    print("Driver: ", driver_path, "(found)" if driver_path.is_file() else "(MISSING)")
    print("Browser:", browser_path, "(found)" if browser_path.is_file() else "(MISSING)")
    if not (driver_path.is_file() and browser_path.is_file()):
        problems.append("a file is missing. Run step 3 again, then this cell.")
    if cache not in driver_path.parents:
        problems.append("that driver isn't Selenium Manager's. Delete it, or run step 2, then steps 3 and 4 again.")
    for warning in manager_warnings:
        print("Selenium Manager warned:", warning)

if problems:
    for problem in problems:
        print("FIX:", problem)
elif manager_warnings:
    print("OK, with the warning above. Go on to step 5; if it fails, the warning is the likely cause.")
else:
    print("OK: a browser, and a driver that matches it.")

## 5 · Start Chrome, visit a page, close it

This is the line that starts the browser in chapter 8: `webdriver.Chrome()`. A Chrome window opens. Leave it alone: this cell drives it, and closes it at the end. Chrome for Testing shows a bar saying it's only for automated testing, which is expected.

In [ ]:
from selenium import webdriver

driver = None
try:
    driver = webdriver.Chrome()
    print("Browser:", driver.capabilities["browserVersion"])
    print("Driver: ", driver.capabilities["chrome"]["chromedriverVersion"].split()[0])
    driver.get("https://quotes.toscrape.com/")
    print("Page title:", driver.title)
    print("OK: Selenium started Chrome, visited a page, and read its title. You're ready for Wednesday.")
except Exception as error:
    message = str(error)
    print(message)
    if "Unable to obtain" in message:
        print("FIX: Selenium couldn't get a driver. Run steps 2 to 4, then this cell again.")
    elif "only supports Chrome version" in message:
        print("FIX: an old driver was used. Run steps 2 to 4 (step 2 tells Selenium to skip it), then this cell again.")
    elif "Chrome instance exited" in message:
        print("FIX: Chrome closed as it started. On Ubuntu 24.04 or later, install Google Chrome from")
        print("https://www.google.com/chrome/ and run steps 3 to 5 again. Otherwise, bring this to the lab.")
    else:
        print("FIX: bring everything this cell printed to Wednesday's lab.")
finally:
    if driver is not None:
        driver.quit()  # always close the browser: each one is a whole Chrome, using memory

## What now

**If step 5 said OK, you're ready for Wednesday.** Here is what stays from this notebook and what doesn't:

- **What stays.** Selenium stays installed in this Python. The driver, and Chrome for Testing if it downloaded, stay in Selenium Manager's cache, so later runs reuse them.
- **What doesn't.** Step 2's settings last only until the kernel restarts. Chapter 8's notebook makes the same settings in its own step 1, under "Before the First Browser", so run its cells in order.

**Why five steps?** `webdriver.Chrome()` does three jobs in one line: it reads the settings, runs Selenium Manager, and starts the browser. When that line fails, its error rarely says which job went wrong. Steps 2, 3, and 5 do those jobs one at a time, and step 4 checks Selenium Manager's work before the browser starts. A failure then shows up in the cell for the job that failed. Chapter 8's "Before the First Browser" explains each step, and its "When Selenium Manager Fails" lists more causes and fixes.

**Still stuck?** Bring everything the failing cell printed, from its first line to its last, to Wednesday's lab.